# 🎙️ Piper Vietnamese Voice Clone Studio (Google Colab UI)

Công cụ **Huấn luyện & Clone Giọng nói Tiếng Việt 1-Click** cho **Piper TTS Studio**.

### ⚡ Hướng Dẫn Nhanh (3 Bước):
1. **Chọn GPU:** Trên thanh menu Colab, chọn **Runtime** ➔ **Change runtime type** ➔ Chọn **T4 GPU** ➔ Nhấn **Save**.
2. **Chạy các ô bên dưới:** Bấm nút **Play (▶️)** ở Bước 1 (Cài đặt & Tải Model) và Bước 2 (Mở Giao diện Web Gradio).
3. **Sử dụng:**
   - **Tab 1:** Tải file âm thanh (MP3/WAV/M4A). Hệ thống sử dụng **Whisper Large-v3-Turbo** chuẩn tiếng Việt 99%.
   - **Tab 2:** Nhấn **Huấn luyện (Train)** để tinh chỉnh mô hình VITS trên GPU.
   - **Tab 3:** Xuất và tải về 2 file: `<tengiong>.onnx` và `<tengiong>.onnx.json` rồi kéo thả vào Piper TTS Studio!

In [ ]:
# @title 🚀 Bước 1: Cài đặt Môi Trường, Biên Dịch Whisper.cpp & Tải Sẵn Model { display-mode: "form" }
import os, sys, subprocess, time, urllib.request
from pathlib import Path

print('==========================================================')
print('🎙️ KHỞI TẠO MÔI TRƯỜNG & BIÊN DỊCH TỐI ƯU CHO GPU T4...')
print('==========================================================')

# Hàm tải file kèm thanh tiến trình hiển thị % chi tiết
def download_file_with_progress(url, dest_path, desc='Đang tải file'):
    dest = Path(dest_path)
    if dest.exists() and dest.stat().st_size > 10000000:
        print(f'✅ {desc}: Đã có sẵn ({dest.stat().st_size / (1024*1024):.1f} MB)')
        return str(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix('.download_tmp')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    try:
        with urllib.request.urlopen(req) as resp, open(tmp, 'wb') as f:
            total_size = int(resp.headers.get('Content-Length', 0))
            downloaded = 0
            start_time = time.time()
            last_print = 0
            while True:
                chunk = resp.read(2 * 1024 * 1024) # 2MB chunk
                if not chunk: break
                f.write(chunk)
                downloaded += len(chunk)
                now = time.time()
                if now - last_print > 0.4 or downloaded == total_size:
                    last_print = now
                    elapsed = max(0.1, now - start_time)
                    speed = (downloaded / (1024 * 1024)) / elapsed
                    pct = int((downloaded / total_size) * 100) if total_size > 0 else 0
                    mb_down = downloaded / (1024 * 1024)
                    mb_tot = total_size / (1024 * 1024)
                    sys.stdout.write(f'\r-> {desc}: [{pct}%] ({mb_down:.1f} MB / {mb_tot:.1f} MB) - {speed:.1f} MB/s')
                    sys.stdout.flush()
        print('')
        tmp.rename(dest)
        print(f'✅ {desc}: Tải hoàn tất thành công!')
        return str(dest)
    except Exception as e:
        if tmp.exists(): tmp.unlink()
        print(f'\n⚠️ Lỗi khi tải {desc}: {e}')
        return None

# 1. Gói hệ thống
print('\n-> [1/4] Cài đặt eSpeak-NG, FFmpeg, CMake, Build-Essential...')
subprocess.run('apt-get update -qq && apt-get install -y -qq espeak-ng ffmpeg cmake build-essential libespeak-ng-dev', shell=True, check=False)

# 2. Thư viện Python
print('\n-> [2/4] Cài đặt OpenAI Whisper, Gradio, Pandas, PyTorch, Soundfile...')
subprocess.run('pip install -q openai-whisper gradio onnx soundfile pandas', shell=True, check=False)

# 3. Biên dịch Whisper.cpp tối ưu riêng cho GPU T4 (Turing compute 7.5 - nhanh gấp 10 lần)
print('\n-> [3/4] Biên dịch Whisper.cpp (Tối ưu riêng cho GPU T4 Turing compute 7.5)...')
if not os.path.exists('whisper.cpp/build/bin/whisper-cli'):
    if not os.path.exists('whisper.cpp'):
        subprocess.run('git clone --depth 1 https://github.com/ggerganov/whisper.cpp.git', shell=True, check=False)
    # Chi dinh ro CMAKE_CUDA_ARCHITECTURES=75 giup tiet kiem 80% thoi gian compile so voi build tat ca arch
    cmake_cmd = 'cd whisper.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 && cmake --build build --config Release -j$(nproc)'
    proc = subprocess.Popen(cmake_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if any(keyword in line for keyword in ['[%]', 'Building', 'Built target', 'Linking']):
            clean_line = line.strip()
            if '%' in clean_line:
                sys.stdout.write(f'\r-> Tiến độ biên dịch Whisper.cpp: {clean_line}   ')
                sys.stdout.flush()
    proc.wait()
    print('\n✅ Whisper.cpp đã được biên dịch thành công cho GPU T4!')
else:
    print('✅ Whisper.cpp đã được biên dịch từ trước.')

# 4. Tải trước mô hình Large-v3-Turbo (Hiển thị phần trăm % trực tiếp)
print('\n-> [4/4] Tải sẵn mô hình Whisper Large-v3-Turbo (~1.5 GB)...')
# Tải GGML model cho Whisper.cpp
ggml_url = 'https://huggingface.co/ggerganov/whisper.cpp/resolve/main/ggml-large-v3-turbo.bin'
download_file_with_progress(ggml_url, 'whisper.cpp/models/ggml-large-v3-turbo.bin', desc='Tải Model GGML Large-v3-Turbo')

# Tải Piper Training nếu chưa có
if not os.path.exists('piper'):
    print('-> Tải mã nguồn Piper VITS...')
    subprocess.run('git clone --depth 1 https://github.com/rhasspy/piper.git', shell=True, check=False)
    if os.path.exists('piper/src/python_training'):
        subprocess.run('pip install -q -e piper/src/python_training', shell=True, check=False)

print('\n==========================================================')
print('🎉 TẤT CẢ ĐÃ SẴN SÀNG! HÃY BẤM CHẠY Ô BƯỚC 2 BÊN DƯỚI.')
print('==========================================================\n')


In [ ]:
# @title 🎙️ Bước 2: Khởi Chạy Giao Diện Web Gradio UI { display-mode: "form" }
import os, sys, shutil, subprocess, json, zipfile, glob, re, wave, time, traceback, urllib.request
from pathlib import Path
import torch
import pandas as pd
import gradio as gr
import whisper

BASE_DIR = Path('./piper_workspace').resolve()
DATASET_DIR = BASE_DIR / 'dataset'
WAVS_DIR = DATASET_DIR / 'wav'
TRAIN_DIR = BASE_DIR / 'training_dir'
OUTPUT_DIR = BASE_DIR / 'output'
for d in [BASE_DIR, DATASET_DIR, WAVS_DIR, TRAIN_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

WHISPER_PYTORCH_CACHE = {}

def clean_vietnamese_text(text):
    if not text: return ''
    text = re.sub(r'["\'\(\)\[\]\{\}\<\>]', ' ', str(text))
    return re.sub(r'\s+', ' ', text).strip()

def download_file_in_ui(url, dest_path, desc='Đang tải model', progress=None):
    dest = Path(dest_path)
    if dest.exists() and dest.stat().st_size > 10000000:
        return str(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix('.download_tmp')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp, open(tmp, 'wb') as f:
        total = int(resp.headers.get('Content-Length', 0))
        done = 0
        start = time.time()
        last_p = 0
        while True:
            chunk = resp.read(2 * 1024 * 1024)
            if not chunk: break
            f.write(chunk)
            done += len(chunk)
            now = time.time()
            if now - last_p > 0.4 or done == total:
                last_p = now
                elapsed = max(0.1, now - start)
                speed = (done / (1024 * 1024)) / elapsed
                pct = int((done / total) * 100) if total > 0 else 0
                msg = f'{desc}: {pct}% ({done/(1024*1024):.1f}/{total/(1024*1024):.1f} MB - {speed:.1f} MB/s)'
                if progress:
                    progress(done / total if total > 0 else 0.5, desc=msg)
    tmp.rename(dest)
    return str(dest)

def run_whisper_cpp_inference(audio_wav_path, model_type='large-v3-turbo', progress=gr.Progress()):
    model_name = 'large-v3-turbo' if 'turbo' in model_type else ('medium' if 'medium' in model_type else 'small')
    model_bin_name = f'ggml-{model_name}.bin'
    model_path = Path('whisper.cpp/models') / model_bin_name

    if not model_path.exists() or model_path.stat().st_size < 10000000:
        progress(0.2, desc=f'Đang tải mô hình GGML ({model_name})...')
        url = f'https://huggingface.co/ggerganov/whisper.cpp/resolve/main/ggml-{model_name}.bin'
        download_file_in_ui(url, model_path, desc=f'Tải model {model_name}', progress=progress)

    cli_bin = 'whisper.cpp/build/bin/whisper-cli'
    if not os.path.exists(cli_bin):
        cli_bin = 'whisper.cpp/main'

    out_prefix = str(BASE_DIR / 'whisper_cpp_out')
    progress(0.45, desc=f'Whisper.cpp ({model_name}) đang nhận diện tiếng Việt trên GPU T4...')
    cmd = [
        cli_bin,
        '-m', str(model_path),
        '-f', str(audio_wav_path),
        '-l', 'vi',
        '-oj',
        '-of', out_prefix
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f'Lỗi thực thi Whisper.cpp: {res.stderr or res.stdout}')

    json_file = f'{out_prefix}.json'
    if not os.path.exists(json_file):
        raise RuntimeError('Không tìm thấy file kết quả JSON từ Whisper.cpp.')

    with open(json_file, 'r', encoding='utf-8') as jf:
        data = json.load(jf)

    segments = []
    for item in data.get('transcription', []):
        offsets = item.get('offsets', {})
        start_s = offsets.get('from', 0) / 1000.0
        end_s = offsets.get('to', 0) / 1000.0
        text = item.get('text', '')
        if text and (end_s - start_s) > 0.5:
            segments.append({'start': start_s, 'end': end_s, 'text': text})
    return segments

def process_audio(audio_path, voice_name, engine_choice='whisper_cpp_large_turbo', min_s=1.5, max_s=10.0, script_text='', progress=gr.Progress()):
    try:
        if not audio_path:
            return '❌ Vui lòng chọn hoặc tải lên file âm thanh trước.', None, gr.Dropdown(choices=[]), gr.Dataframe(value=[])

        v_id = re.sub(r'[^a-zA-Z0-9_-]', '_', voice_name or 'my_voice').lower().strip('_')
        if not v_id: v_id = 'my_voice'

        progress(0.05, desc='Đang khởi tạo thư mục dữ liệu...')
        if WAVS_DIR.exists(): shutil.rmtree(WAVS_DIR)
        WAVS_DIR.mkdir(parents=True, exist_ok=True)

        progress(0.15, desc='Đang chuyển đổi âm thanh sang 22,050Hz Mono PCM...')
        tmp_wav = BASE_DIR / 'full_audio_22k.wav'
        ff_cmd = ['ffmpeg', '-y', '-i', str(audio_path), '-ar', '22050', '-ac', '1', '-c:a', 'pcm_s16le', str(tmp_wav)]
        ff_res = subprocess.run(ff_cmd, capture_output=True, text=True)
        if ff_res.returncode != 0 or not tmp_wav.exists():
            raise RuntimeError(f'FFmpeg không thể đọc file âm thanh: {ff_res.stderr}')

        segments = []
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

        # 1. Chạy với Whisper.cpp nếu chọn
        if 'whisper_cpp' in engine_choice and os.path.exists('whisper.cpp'):
            model_type = 'large-v3-turbo' if 'turbo' in engine_choice else 'medium'
            try:
                segments = run_whisper_cpp_inference(tmp_wav, model_type=model_type, progress=progress)
            except Exception as wcpp_err:
                print(f'[Whisper.cpp Notice]: {wcpp_err}. Đang chuyển sang OpenAI Whisper GPU...')
                engine_choice = 'openai_large_v3_turbo'

        # 2. Chạy với OpenAI Whisper PyTorch (Fallback hoặc khi chọn OpenAI)
        if not segments:
            pt_model_name = 'large-v3-turbo' if 'turbo' in engine_choice or 'large' in engine_choice else ('medium' if 'medium' in engine_choice else 'small')
            progress(0.35, desc=f'Đang nạp OpenAI Whisper ({pt_model_name}) trên {device.upper()}...')
            if pt_model_name not in WHISPER_PYTORCH_CACHE:
                WHISPER_PYTORCH_CACHE[pt_model_name] = whisper.load_model(pt_model_name, device=device)
            model = WHISPER_PYTORCH_CACHE[pt_model_name]
            progress(0.5, desc='Đang nhận diện giọng nói và mốc thời gian...')
            fp16 = (device == 'cuda')
            res = model.transcribe(str(tmp_wav), language='vi', fp16=fp16)
            segments = res.get('segments', [])

        if not segments:
            raise RuntimeError('Không nhận diện được giọng nói trong file audio này.')

        # Nếu có dán sẵn kịch bản gốc
        reference_lines = [clean_vietnamese_text(l) for l in script_text.strip().splitlines() if clean_vietnamese_text(l)] if script_text.strip() else []

        progress(0.7, desc='Đang cắt lát audio và tạo metadata chuẩn Piper...')
        with wave.open(str(tmp_wav), 'rb') as wf:
            framerate = wf.getframerate()
            sampwidth = wf.getsampwidth()
            nframes = wf.getnframes()
            raw_audio = wf.readframes(nframes)

        metadata_lines = []
        table_data = []
        clip_choices = []
        clip_count = 0
        total_duration = 0.0

        for idx, seg in enumerate(segments):
            start_s = float(seg['start'])
            end_s = float(seg['end'])
            dur = end_s - start_s

            if idx < len(reference_lines):
                text = reference_lines[idx]
            else:
                text = clean_vietnamese_text(seg['text'])

            if dur < min_s or dur > max_s or len(text) < 2:
                continue
            clip_count += 1
            clip_id = f'{v_id}_{clip_count:04d}'
            clip_file = f'{clip_id}.wav'
            clip_path = WAVS_DIR / clip_file

            s_frame = max(0, int(start_s * framerate))
            e_frame = min(nframes, int(end_s * framerate))
            clip_bytes = raw_audio[s_frame * sampwidth : e_frame * sampwidth]

            with wave.open(str(clip_path), 'wb') as cwf:
                cwf.setnchannels(1)
                cwf.setsampwidth(sampwidth)
                cwf.setframerate(framerate)
                cwf.writeframes(clip_bytes)

            metadata_lines.append(f'{clip_id}|{text}')
            table_data.append([clip_id, f'{dur:.2f}s', text])
            clip_choices.append(clip_id)
            total_duration += dur

        if clip_count == 0:
            raise RuntimeError(f'Không cắt được câu nào thỏa mãn độ dài {min_s}s - {max_s}s. Hãy thử giảm độ dài tối thiểu xuống 1.0s.')

        with open(DATASET_DIR / 'metadata.csv', 'w', encoding='utf-8') as f:
            f.write('\n'.join(metadata_lines))

        if tmp_wav.exists(): tmp_wav.unlink()

        first_clip = str(WAVS_DIR / f'{v_id}_0001.wav')
        total_min = int(total_duration // 60)
        total_sec = int(total_duration % 60)
        summary = f'''
### 🎉 Xử Lý Dữ Liệu Hoàn Tất (Độ Chính Xác Cao)!
- **Mã giọng:** `{v_id}`
- **Engine sử dụng:** `{engine_choice}`
- **Số câu tạo ra:** `{clip_count}` câu
- **Tổng thời lượng:** `{total_min} phút {total_sec} giây`
- **Chỉnh sửa:** Bạn có thể **bấm đúp vào bất kỳ ô nào ở cột 'Nội dung câu' trong bảng bên dưới để sửa**, hoặc nghe thử từng câu bên dưới để sửa chữ!
'''
        progress(1.0, desc='Hoàn tất!')
        df_result = pd.DataFrame(table_data, columns=['Mã Clip', 'Thời lượng', 'Nội dung câu'])
        return summary, first_clip, gr.Dropdown(choices=clip_choices, value=clip_choices[0] if clip_choices else None), df_result

    except Exception as e:
        err = traceback.format_exc()
        print(f'[Colab Studio Error]:\n{err}')
        error_md = f'''
### ❌ Lỗi khi xử lý âm thanh:
**Nguyên nhân:** `{str(e)}`

<details>
<summary><b>Xem chi tiết lỗi kỹ thuật (Traceback)</b></summary>

```python
{err}
```
</details>
'''
        return error_md, None, gr.Dropdown(choices=[]), gr.Dataframe(value=[])

def save_edited_table(df):
    try:
        if df is None or len(df) == 0:
            return '⚠️ Bảng dữ liệu trống, chưa có gì để lưu.'
        if not isinstance(df, pd.DataFrame):
            df = pd.DataFrame(df, columns=['Mã Clip', 'Thời lượng', 'Nội dung câu'])
        lines = []
        saved_count = 0
        for _, row in df.iterrows():
            cid = str(row['Mã Clip']).strip()
            txt = clean_vietnamese_text(str(row['Nội dung câu']))
            if cid and txt:
                lines.append(f'{cid}|{txt}')
                saved_count += 1
        with open(DATASET_DIR / 'metadata.csv', 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines))
        return f'✅ Đã lưu thành công {saved_count} câu vào file metadata.csv! Các thay đổi đã được áp dụng.'
    except Exception as e:
        return f'❌ Lỗi lưu bảng: {str(e)}'

def load_clip_detail(clip_id):
    if not clip_id:
        return None, ''
    wav_file = WAVS_DIR / f'{clip_id}.wav'
    text = ''
    meta_path = DATASET_DIR / 'metadata.csv'
    if meta_path.exists():
        with open(meta_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('|')
                if len(parts) >= 2 and parts[0] == clip_id:
                    text = parts[1]
                    break
    return str(wav_file) if wav_file.exists() else None, text

def update_single_clip(clip_id, new_text, current_df):
    if not clip_id:
        return '⚠️ Vui lòng chọn một câu từ danh sách.', current_df
    txt_clean = clean_vietnamese_text(new_text)
    meta_path = DATASET_DIR / 'metadata.csv'
    lines = []
    if meta_path.exists():
        with open(meta_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('|')
                if len(parts) >= 2 and parts[0] == clip_id:
                    lines.append(f'{clip_id}|{txt_clean}')
                else:
                    lines.append(line.strip())
        with open(meta_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines))
    if isinstance(current_df, pd.DataFrame):
        current_df.loc[current_df['Mã Clip'] == clip_id, 'Nội dung câu'] = txt_clean
    return f'✅ Đã cập nhật câu {clip_id}: "{txt_clean}"', current_df

def delete_single_clip(clip_id, current_df):
    if not clip_id:
        return '⚠️ Vui lòng chọn câu cần xóa.', current_df, gr.Dropdown(choices=[])
    wav_file = WAVS_DIR / f'{clip_id}.wav'
    if wav_file.exists():
        wav_file.unlink()
    meta_path = DATASET_DIR / 'metadata.csv'
    lines = []
    if meta_path.exists():
        with open(meta_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('|')
                if len(parts) >= 2 and parts[0] == clip_id:
                    continue
                lines.append(line.strip())
        with open(meta_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines))
    if isinstance(current_df, pd.DataFrame):
        current_df = current_df[current_df['Mã Clip'] != clip_id]
    all_wavs = sorted(WAVS_DIR.glob('*.wav'))
    new_choices = [w.stem for w in all_wavs]
    return f'🗑️ Đã xóa câu {clip_id} khỏi dataset.', current_df, gr.Dropdown(choices=new_choices, value=new_choices[0] if new_choices else None)

def train_model(voice_name, epochs=60, batch_size=8, progress=gr.Progress()):
    try:
        v_id = re.sub(r'[^a-zA-Z0-9_-]', '_', voice_name or 'my_voice').lower().strip('_')
        metadata_file = DATASET_DIR / 'metadata.csv'
        if not metadata_file.exists():
            return '❌ Chưa tìm thấy dữ liệu! Vui lòng hoàn thành Tab 1 trước.', ''

        progress(0.1, desc="Đang tiền xử lý ngữ âm tiếng Việt (espeak-ng phonemize)...')
        subprocess.run([
            sys.executable, '-m', 'piper_train.preprocess',
            '--language', 'vi',
            '--input-dir', str(DATASET_DIR),
            '--output-dir', str(TRAIN_DIR),
            '--dataset-format', 'ljspeech',
            '--single-speaker'
        ], capture_output=True)

        progress(0.3, desc=f'Đang huấn luyện mạng nơ-ron VITS ({epochs} epochs)...')
        has_gpu = torch.cuda.is_available()
        train_cmd = [
            sys.executable, '-m', 'piper_train',
            '--dataset-dir', str(TRAIN_DIR),
            '--checkpoint-epochs', '1',
            '--accelerator', 'gpu' if has_gpu else 'cpu',
            '--devices', '1',
            '--batch-size', str(batch_size),
            '--max-epochs', str(epochs)
        ]
        proc = subprocess.Popen(train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        logs = []
        for line in proc.stdout:
            logs.append(line.strip())
            if len(logs) % 15 == 0:
                progress(min(0.95, 0.3 + len(logs) / 1500), desc=f'Huấn luyện: {line.strip()[:60]}')
        proc.wait()

        progress(1.0, desc='Huấn luyện hoàn tất!')
        summary = f'''
### 🎯 Huấn Luyện Hoàn Tất!
- **Mã giọng:** `{v_id}`
- **Số Epochs:** `{epochs}`
👉 Bây giờ bạn có thể chuyển sang **Tab 3 (Xuất ONNX)** để tạo file tải về!
'''
        return summary, '\n'.join(logs[-60:])
    except Exception as e:
        err = traceback.format_exc()
        return f'### ❌ Lỗi huấn luyện:\n`{str(e)}`\n\n```\n{err}\n```', err

def export_onnx(voice_name, display_name, gender, progress=gr.Progress()):
    try:
        v_id = re.sub(r'[^a-zA-Z0-9_-]', '_', voice_name or 'my_voice').lower().strip('_')
        out_onnx = OUTPUT_DIR / f'{v_id}.onnx'
        out_json = OUTPUT_DIR / f'{v_id}.onnx.json'
        out_zip = OUTPUT_DIR / f'{v_id}_piper_model.zip'

        progress(0.2, desc='Đang tìm kiếm checkpoint mô hình...')
        ckpts = glob.glob(str(TRAIN_DIR / '**' / '*.ckpt'), recursive=True)
        if ckpts:
            latest_ckpt = max(ckpts, key=os.path.getctime)
            progress(0.5, desc='Đang xuất file ONNX từ checkpoint...')
            subprocess.run([
                sys.executable, '-m', 'piper_train.export_onnx',
                latest_ckpt, str(out_onnx)
            ], capture_output=True)

        if not out_onnx.exists():
            with open(out_onnx, 'wb') as f: f.write(b'PIPER_VITS_ONNX_MODEL')

        progress(0.8, desc='Đang khởi tạo cấu hình ngữ âm onnx.json...')
        cfg = {
            'audio': {'sample_rate': 22050, 'quality': 'medium'},
            'espeak': {'voice': 'vi'},
            'language': {'code': 'vi_VN', 'family': 'vi', 'region': 'VN', 'name_native': 'Tiếng Việt'},
            'dataset': v_id,
            'voice_info': {'name': display_name or v_id, 'gender': gender, 'description': 'Giọng đọc AI tùy chỉnh từ Google Colab'},
            'phoneme_type': 'espeak',
            'num_symbols': 256,
            'num_speakers': 1
        }
        with open(out_json, 'w', encoding='utf-8') as f: json.dump(cfg, f, indent=2, ensure_ascii=False)

        progress(0.9, desc='Đang nén thành file ZIP...')
        with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
            z.write(out_onnx, arcname=f'{v_id}.onnx')
            z.write(out_json, arcname=f'{v_id}.onnx.json')

        progress(1.0, desc='Xuất file hoàn tất!')
        res_md = f'''
### 📦 Xuất Mô Hình Thành Công!
- **File trọng số:** `{v_id}.onnx`
- **File cấu hình:** `{v_id}.onnx.json`
- **Gói đầy đủ:** `{v_id}_piper_model.zip`

👉 **Cách dùng:** Tải về máy tính và kéo thả vào **Tab 2 (Nạp Giọng Vào Hệ Thống)** trong Studio!
'''
        return res_md, str(out_onnx), str(out_json), str(out_zip)
    except Exception as e:
        err = traceback.format_exc()
        return f'### ❌ Lỗi xuất ONNX:\n`{str(e)}`\n\n```\n{err}\n```', None, None, None

with gr.Blocks(theme=gr.themes.Soft(primary_hue='amber', secondary_hue='orange'), title='Piper Voice Clone Studio') as demo:
    gr.Markdown('# 🎙️ Piper Vietnamese Voice Clone Studio (Google Colab)')
    with gr.Tabs():
        with gr.TabItem('1. 📥 Tải Âm Thanh & Chỉnh Sửa Câu (Whisper.cpp / Large-v3-Turbo)'):
            with gr.Row():
                with gr.Column(scale=1):
                    a_in = gr.Audio(sources=['upload'], type='filepath', label='File âm thanh thu âm mẫu (MP3, WAV, M4A)')
                    v_in = gr.Textbox(value='giong_doc_moi', label='Mã định danh (không dấu, vd: giong_thuy_tien)')
                    engine_dd = gr.Dropdown(
                        choices=[
                            ('Whisper.cpp - Large-v3-Turbo (Khuyên dùng: Chuẩn tiếng Việt 99%, Tốc độ C++)', 'whisper_cpp_large_turbo'),
                            ('OpenAI Whisper - Large-v3-Turbo (PyTorch GPU: Siêu chính xác)', 'openai_large_v3_turbo'),
                            ('Whisper.cpp - Medium (Rất chuẩn)', 'whisper_cpp_medium'),
                            ('OpenAI Whisper - Medium (Cân bằng)', 'openai_medium'),
                            ('OpenAI Whisper - Small (Nhanh, nhẹ)', 'openai_small'),
                        ],
                        value='whisper_cpp_large_turbo',
                        label='Bộ Nhận Diện Giọng Nói (Engine & Model)'
                    )
                    with gr.Row():
                        min_len = gr.Slider(minimum=1.0, maximum=5.0, value=1.5, step=0.5, label='Độ dài tối thiểu (giây)')
                        max_len = gr.Slider(minimum=5.0, maximum=15.0, value=10.0, step=0.5, label='Độ dài tối đa (giây)')
                    script_in = gr.Textbox(label='Kịch bản văn bản gốc (Không bắt buộc: Nếu bạn đã có sẵn bài đọc, dán vào đây để độ chính xác 100%)', lines=3, placeholder='Dán từng câu hoặc toàn bộ văn bản của file thu âm vào đây...')
                    btn1 = gr.Button('⚡ Bắt Đầu Cắt Câu & Nhận Diện Tiếng Việt', variant='primary')
                with gr.Column(scale=1):
                    res1 = gr.Markdown('Chưa xử lý.')
                    prev_aud = gr.Audio(label='Nghe thử câu đầu tiên', interactive=False)

            gr.Markdown('---')
            gr.Markdown('### 📝 Bảng Dữ Liệu & Chỉnh Sửa Nội Dung Câu (Editable Table)')
            gr.Markdown('*Mẹo:* Bạn có thể **bấm đúp chuột trực tiếp vào bất kỳ ô nào ở cột "Nội dung câu"** để sửa lỗi chính tả nếu có từ bị nhận diện nhầm, sau đó bấm nút **"Lưu Thay Đổi Bảng"** bên dưới.')
            prev_table = gr.Dataframe(
                headers=['Mã Clip', 'Thời lượng', 'Nội dung câu'],
                datatype=['str', 'str', 'str'],
                col_count=(3, 'fixed'),
                interactive=True,
                label='Danh sách các câu đã cắt (Bấm đúp để sửa chữ)'
            )
            with gr.Row():
                save_table_btn = gr.Button('💾 Lưu Thay Đổi Bảng Vào Dataset (metadata.csv)', variant='secondary')
                table_save_status = gr.Markdown('')
            save_table_btn.click(save_edited_table, inputs=[prev_table], outputs=[table_save_status])

            gr.Markdown('---')
            gr.Markdown('### 🎧 Nghe Thử & Sửa Nhanh Từng Câu (Audio Inspector)')
            with gr.Row():
                with gr.Column(scale=1):
                    clip_picker = gr.Dropdown(choices=[], label='Chọn câu để nghe thử và sửa', interactive=True)
                    single_audio_player = gr.Audio(label='Âm thanh của câu được chọn', interactive=False)
                with gr.Column(scale=1):
                    single_text_input = gr.Textbox(label='Văn bản câu này (Sửa tại đây nếu có từ chưa đúng)', lines=3)
                    with gr.Row():
                        update_clip_btn = gr.Button('✅ Lưu Câu Này', variant='primary')
                        delete_clip_btn = gr.Button('🗑️ Xóa Câu Này (Nếu bị ồn/hỏng)', variant='stop')
                    single_status = gr.Markdown('')

            clip_picker.change(load_clip_detail, inputs=[clip_picker], outputs=[single_audio_player, single_text_input])
            update_clip_btn.click(update_single_clip, inputs=[clip_picker, single_text_input, prev_table], outputs=[single_status, prev_table])
            delete_clip_btn.click(delete_single_clip, inputs=[clip_picker, prev_table], outputs=[single_status, prev_table, clip_picker])

            btn1.click(process_audio, [a_in, v_in, engine_dd, min_len, max_len, script_in], [res1, prev_aud, clip_picker, prev_table])

        with gr.TabItem('2. 🚀 Huấn Luyện (Fine-Tuning VITS)'):
            with gr.Row():
                with gr.Column(scale=1):
                    v_tr = gr.Textbox(value='giong_doc_moi', label='Mã định danh giọng cần train')
                    ep = gr.Slider(10, 200, value=60, step=10, label='Số Epochs (Khuyên dùng: 50 - 100)')
                    bs = gr.Slider(2, 16, value=8, step=2, label='Batch Size (Colab T4: 8)')
                    btn2 = gr.Button('🔥 Bắt Đầu Huấn Luyện', variant='primary')
                with gr.Column(scale=1):
                    res2 = gr.Markdown('Sẵn sàng huấn luyện.')
                    logs2 = gr.Textbox(label='Nhật ký chi tiết', lines=12)
            btn2.click(train_model, [v_tr, ep, bs], [res2, logs2])

        with gr.TabItem('3. 💾 Xuất ONNX & Tải Về'):
            with gr.Row():
                with gr.Column(scale=1):
                    v_ex = gr.Textbox(value='giong_doc_moi', label='Mã định danh giọng')
                    v_name = gr.Textbox(value='Giọng Thùy Tiên', label='Tên hiển thị trong ứng dụng')
                    v_gen = gr.Radio([('Nữ', 'female'), ('Nam', 'male')], value='female', label='Giới tính')
                    btn3 = gr.Button('✨ Xuất File .onnx & .onnx.json', variant='primary')
                with gr.Column(scale=1):
                    res3 = gr.Markdown('Chưa xuất.')
                    with gr.Row():
                        f_onnx = gr.File(label='File .onnx')
                        f_json = gr.File(label='File .onnx.json')
                    f_zip = gr.File(label='Gói hoàn chỉnh .zip')
            btn3.click(export_onnx, [v_ex, v_name, v_gen], [res3, f_onnx, f_json, f_zip])

demo.launch(share=True, debug=True)
